This is a starter notebook for the project, you'll have to import the libraries you'll need, you can find a list of the ones available in this workspace in the requirements.txt file in this workspace. 

In [13]:
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["OPENAI_API_BASE"] = "https://openai.vocareum.com/v1"

# Essential
import os
import csv
import shutil
from datetime import datetime

# LangChain 
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationSummaryMemory, ConversationBufferMemory
from langchain.chains import LLMChain, ConversationChain
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.schema import Document
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.memory import ChatMessageHistory
from langchain.memory import ConversationBufferMemory

# OpenAI
import openai

model_name = 'gpt-3.5-turbo'

### Generating Real Estate Listings
Generating real estate listings using a data model and prompt template approach.

In [14]:
class RealEstateListing:
    def __init__(
        self,
        neighborhood: str,
        price: int,
        bedrooms: int,
        bathrooms: int,
        size_sqft: int,
    ):
        # Property details
        self.neighborhood = neighborhood  # Location/area name
        self.price = price  # Price in  €
        self.bedrooms = bedrooms  # Number of bedrooms
        self.bathrooms = bathrooms  # Number of bathrooms
        self.size_sqft = size_sqft  # Property size in square feet
        self.description = ""  # Generated property description
        self.neighborhood_description = ""  # Generated area description


# Prompt Template for LLM

prompt_template = """
Create a real estate listing with these details:
- Neighborhood: {neighborhood}
- Price: €{price:,}
- Bedrooms: {bedrooms}
- Bathrooms: {bathrooms}
- Size: {size_sqft:,} sqft

Include:
1. A property description (2-3 sentences)
2. A neighborhood description (1-2 sentences)
"""


# Create 10 Berlin property listings
listings = [
    RealEstateListing(
        neighborhood="Mitte", price=550000, bedrooms=2, bathrooms=1, size_sqft=950
    ),
    RealEstateListing(
        neighborhood="Prenzlauer Berg",
        price=480000,
        bedrooms=3,
        bathrooms=2,
        size_sqft=1250,
    ),
    RealEstateListing(
        neighborhood="Kreuzberg", price=420000, bedrooms=2, bathrooms=1, size_sqft=850
    ),
    RealEstateListing(
        neighborhood="Friedrichshain",
        price=380000,
        bedrooms=1,
        bathrooms=1,
        size_sqft=650,
    ),
    RealEstateListing(
        neighborhood="Charlottenburg",
        price=620000,
        bedrooms=3,
        bathrooms=2,
        size_sqft=1450,
    ),
    RealEstateListing(
        neighborhood="Neukölln", price=320000, bedrooms=2, bathrooms=1, size_sqft=900
    ),
    RealEstateListing(
        neighborhood="Schöneberg", price=510000, bedrooms=2, bathrooms=2, size_sqft=1100
    ),
    RealEstateListing(
        neighborhood="Tempelhof", price=370000, bedrooms=3, bathrooms=1, size_sqft=1200
    ),
    RealEstateListing(
        neighborhood="Wilmersdorf",
        price=590000,
        bedrooms=3,
        bathrooms=2,
        size_sqft=1350,
    ),
    RealEstateListing(
        neighborhood="Moabit", price=350000, bedrooms=1, bathrooms=1, size_sqft=750
    ),
]


In [15]:
# ==================== Chat Completion ==================== 

def create_chat_completion(prompt: str) -> str:
    """
    Generate real estate listing using OpenAI API.
    
    Args:
        prompt: Formatted prompt with property details
        
    Returns:
        Generated listing text
    """
    try:
        response = openai.ChatCompletion.create(
            model=model_name,
            messages=[
                {
                    "role": "system",
                    "content": "You are a real estate agent. You are writing about real estate listing."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=1,
            max_tokens=256,
            top_p=1,
            frequency_penalty=0,
            presence_penalty=0
        )
        
        # Return the generated content
        return response.choices[0].message.content
        
    except Exception as e:
        return f"Error generating listing: {str(e)}"

In [16]:
# Save to Listings.txt first
print("📄 Creating Listings.txt...")

with open("Listings.txt", "w", encoding="utf-8") as txtfile:
    txtfile.write("HOMEMATCH PROPERTY LISTINGS\n\n")

    for i, listing in enumerate(listings, 1):
        txtfile.write(f"Listing {i}:\n")
        txtfile.write(f"  Neighborhood: {listing.neighborhood}\n")
        txtfile.write(f"  Price: €{listing.price:,}\n")
        txtfile.write(f"  Bedrooms: {listing.bedrooms}\n")
        txtfile.write(f"  Bathrooms: {listing.bathrooms}\n")
        txtfile.write(f"  Size: {listing.size_sqft:,} sqft\n\n")

        prompt = prompt_template.format(
            neighborhood=listing.neighborhood,
            price=listing.price,
            bedrooms=listing.bedrooms,
            bathrooms=listing.bathrooms,
            size_sqft=listing.size_sqft,
        )

        description = create_chat_completion(prompt)
        if description:
            txtfile.write(f"  Description: {description}\n\n")
            txtfile.write("-" * 50 + "\n\n")

print(f"✅ Listings.txt created with {len(listings)} properties")

csv_filename = "listings.csv"
with open(csv_filename, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(
        ["Neighborhood", "Price", "Bedrooms", "Bathrooms", "Size", "Description"]
    )

    for listing in listings:
        prompt = prompt_template.format(
            neighborhood=listing.neighborhood,
            price=listing.price,
            bedrooms=listing.bedrooms,
            bathrooms=listing.bathrooms,
            size_sqft=listing.size_sqft,
        )

        description = create_chat_completion(prompt)
        if description:
            writer.writerow(
                [
                    listing.neighborhood,
                    listing.price,
                    listing.bedrooms,
                    listing.bathrooms,
                    listing.size_sqft,
                    description,
                ]
            )

print(f"📊 CSV file created: {csv_filename}")

📄 Creating Listings.txt...
✅ Listings.txt created with 10 properties
📊 CSV file created: listings.csv


## Storing Listings in a Vector Database

In [17]:
# Clear old database if any

persist_directory = "./real_estate_db"

if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

print("🧹 Cleared old database")

# Load from CSV for vector database
documents = []
with open(csv_filename, "r", encoding="utf-8") as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        doc_text = f"""
        Property in {row['Neighborhood']}
        Price: €{int(row['Price']):,}
        Bedrooms: {row['Bedrooms']}
        Bathrooms: {row['Bathrooms']}
        Size: {row['Size']} sqft
        
        Description:
        {row['Description']}
        """

        metadata = {
            "neighborhood": row["Neighborhood"],
            "price": int(row["Price"]),
            "bedrooms": int(row["Bedrooms"]),
            "bathrooms": int(row["Bathrooms"]),
            "size_sqft": int(row["Size"]),
        }

        documents.append(Document(page_content=doc_text.strip(), metadata=metadata))

print(f"✅ Loaded {len(documents)} documents from {csv_filename}")


🧹 Cleared old database
✅ Loaded 10 documents from listings.csv


In [18]:
# ==================== CREATE EMBEDDINGS ====================
# Split into chunks
splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = splitter.split_documents(documents)
print(f"✂️  Split into {len(split_docs)} chunks")

# Create vector database
embeddings = OpenAIEmbeddings(
    openai_api_key=openai.api_key, openai_api_base=os.environ.get("OPENAI_API_BASE")
)

vectordb = Chroma.from_documents(
    documents=split_docs, embedding=embeddings, persist_directory=persist_directory
)

vectordb.persist()
print(f"💾 Vector database created at: {persist_directory}")
print(f"📊 Total embeddings: {len(split_docs)}")

✂️  Split into 12 chunks
💾 Vector database created at: ./real_estate_db
📊 Total embeddings: 12


In [19]:
# ==================== CHECK THE DATABASE ====================
query = "modern apartment in Berlin"
results = vectordb.similarity_search(query, k=1)

print(f"\n🔍 Test query: '{query}'")
print(f"📊 Found {len(results)} similar listings:")

for i, result in enumerate(results):
    print(f"\n{i+1}. Neighborhood: {result.metadata.get('neighborhood', 'N/A')}")
    print(f"   Price: €{result.metadata.get('price', 'N/A'):,}")
    print(f"   Bedrooms: {result.metadata.get('bedrooms', 'N/A')}")
    print(f"   Preview: {result.page_content}")


🔍 Test query: 'modern apartment in Berlin'
📊 Found 1 similar listings:

1. Neighborhood: Friedrichshain
   Price: €380,000
   Bedrooms: 1
   Preview: Property in Friedrichshain
        Price: €380,000
        Bedrooms: 1
        Bathrooms: 1
        Size: 650 sqft
        
        Description:
        **Real Estate Listing:**

**Location:** Friedrichshain, Berlin

**Price:** €380,000

**Bedrooms:** 1

**Bathrooms:** 1

**Size:** 650 sqft

**Property Description:**
This charming 1-bedroom, 1-bathroom apartment in Friedrichshain offers a cozy and modern living space ideal for individuals or couples. The open layout and large windows allow natural light to flood the space, creating a bright and airy ambiance. With its contemporary design and convenient location, this property is perfect for those looking for a comfortable urban home in one of Berlin's most vibrant neighborhoods.


###  Building the User Preference Interface

In [20]:
# ==================== Interactive property finder ====================


def interactive_property_finder():
    print("🏡 BERLIN PROPERTY FINDER")
    print("Answer a few questions to get personalized recommendations\n")

    questions = {
        "budget": "What's your budget range? (e.g., 'under €400,000' or '€300,000-500,000')",
        "bedrooms": "How many bedrooms do you need?",
        "priorities": "What are your top 3 priorities? (e.g., 'location, schools, transport')",
        "lifestyle": "Describe your ideal neighborhood atmosphere:",
        "commute": "How important is public transport access? (1-5, 5 being most important)",
    }

    answers = {}
    for key, question in questions.items():
        answer = input(f"\n❓ {question}\n   → ")
        answers[key] = answer

    # Create search query
    query = f"""
    Looking for property in Berlin with:
    Budget: {answers['budget']}
    Bedrooms: {answers['bedrooms']}
    Priorities: {answers['priorities']}
    Neighborhood: {answers['lifestyle']}
    Transport importance: {answers['commute']}/5
    """

    print("\n" + "=" * 60)
    print("🔍 Searching for your perfect property...")

    results = vectordb.similarity_search(query, k=3)

    if results:
        print(f"\n✅ Found {len(results)} properties for you:\n")
        for i, prop in enumerate(results):
            print(
                f"{i+1}. {prop.metadata['neighborhood']} - €{prop.metadata['price']:,}"
            )
    else:
        print("\n😔 No properties found. Try adjusting your criteria.")


In [21]:
interactive_property_finder()

🏡 BERLIN PROPERTY FINDER
Answer a few questions to get personalized recommendations


❓ What's your budget range? (e.g., 'under €400,000' or '€300,000-500,000')
   → €400,000

❓ How many bedrooms do you need?
   → 2

❓ What are your top 3 priorities? (e.g., 'location, schools, transport')
   → location

❓ Describe your ideal neighborhood atmosphere:
   → calm

❓ How important is public transport access? (1-5, 5 being most important)
   → 2

🔍 Searching for your perfect property...

✅ Found 3 properties for you:

1. Schöneberg - €510,000
2. Neukölln - €320,000
3. Kreuzberg - €420,000


 ## Searching Based on Preferences

In [22]:
# LLM
llm = ChatOpenAI(
        model_name=model_name,
        temperature=0.3,
        max_tokens=1000,
        openai_api_base=os.environ["OPENAI_API_BASE"],
        openai_api_key=os.environ["OPENAI_API_KEY"]
    )

use_chain_helper = True
if use_chain_helper:
    rag = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectordb.as_retriever())
    print(rag.run(query))
else:
    # similar_docs = db.similarity_search(query, k=5)
    prompt = PromptTemplate(
        template="{query}\nContext: {context}",
        input_variables=["query", "context"],
    )
    chain = load_qa_chain(llm, prompt = prompt, chain_type="stuff")
    print(chain.run(input_documents=results, query = query))

Here are some modern apartments in Berlin:

1. **Property in Friedrichshain:**
   - Price: €380,000
   - Bedrooms: 1
   - Bathrooms: 1
   - Size: 650 sqft
   - Description: Charming 1-bedroom apartment in Friedrichshain with a cozy and modern living space.

2. **Property in Neukölln:**
   - Price: €320,000
   - Bedrooms: 2
   - Bathrooms: 1
   - Size: 900 sqft
   - Description: Charming 2-bedroom apartment in Neukölln, perfect for those seeking a cozy abode in a bustling area.

3. **Property in Kreuzberg:**
   - Price: €420,000
   - Bedrooms: 2
   - Bathrooms: 1
   - Size: 850 sqft
   - Description: Stylish 2-bedroom apartment in Kreuzberg, offering a cozy and comfortable atmosphere.

4. **Property in Schöneberg:**
   - Price: €510,000
   - Bedrooms: 2
   - Bathrooms: 2
   - Size: 1100 sqft
   - Description: Bright and spacious 2-bedroom home in Schöneberg with modern finishes and a private balcony.


## Personalize listing descriptions

In [23]:
# Create memory with summary capability
personal_questions = [
    "How big do you want your house to be?", 
    "What are 3 most important things for you in choosing this property?", 
    "Which amenities would you like?", 
    "Which transportation options are important to you?",
    "How urban do you want your neighborhood to be?"
]

personal_answers = [
    "A comfortable 2-bedroom, 1-bathroom home with a spacious kitchen and a cozy living room.",
    "A quiet neighborhood and good local schools.",
    "A backyard for gardening and barbecues.",
    "Easy access to public transport.",
    "A balance between suburban tranquility and access to urban amenities like restaurants and theaters."
]

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Preferences
print("📝 Storing buyer preferences...")
for i, (q, a) in enumerate(zip(personal_questions, personal_answers), 1):
    memory.save_context(
        {"input": f"Q{i}: {q}"},
        {"output": f"A{i}: {a}"}
    )
    print(f"  ✓ {q[:40]}...")

# The Memory content
memory_vars = memory.load_memory_variables({})
print(f"\n📊 Memory contains {len(memory_vars['chat_history'])} messages")

def get_memory_context():
    """Return buyer preferences."""
    qa_text = "\n".join([f"Q: {q}\nA: {a}\n" for q, a in zip(personal_questions, personal_answers)])
    
    summary = "BUYER WANTS:\n"
    summary += "• 2-bedroom, 1-bathroom home with spacious kitchen\n"
    summary += "• Quiet neighborhood with good schools\n"
    summary += "• Backyard for gardening and barbecues\n"
    summary += "• Easy access to public transport\n"
    summary += "• Balanced urban-suburban lifestyle\n"
    
    print(f"✅ Sending {len(qa_text)} chars of Q&A to LLM")
    print(f"✅ Sending summary: {summary[:100]}...")
    
    return {
        "recommendation_summary": summary,
        "questions_and_answers": qa_text
    }

📝 Storing buyer preferences...
  ✓ How big do you want your house to be?...
  ✓ What are 3 most important things for you...
  ✓ Which amenities would you like?...
  ✓ Which transportation options are importa...
  ✓ How urban do you want your neighborhood ...

📊 Memory contains 10 messages


## RECOMMENDER

In [24]:
RECOMMENDER_TEMPLATE = """
You are a real estate agent creating personalized property descriptions.

BUYER PREFERENCES SUMMARY:
{recommendation_summary}

DETAILED BUYER Q&A:
{questions_and_answers}

PROPERTY TO PERSONALIZE:
{input}

INSTRUCTIONS:
1. MENTION SPECIFIC BUYER PREFERENCES from above
2. Say things like: "Based on your preference for..." or "Since you mentioned you want..."
3. Highlight how this property matches what the buyer is looking for
4. Keep factual information accurate
5. Make it feel personalized to this specific buyer

PERSONALIZED PROPERTY DESCRIPTION:
"""

PROMPT = PromptTemplate(
    input_variables=["recommendation_summary", "questions_and_answers", "input"],
    template=RECOMMENDER_TEMPLATE,
)

recommender_chain = LLMChain(llm=llm, prompt=PROMPT, verbose=False)
# Process listings with VERIFICATION
for listing in listings:
    # Get context
    context = get_memory_context()
    
    # VERIFY preferences are being sent
    print(f"\n🔍 SENDING TO LLM FOR {listing.neighborhood}:")
    print(f"Q&A contains 'bedroom': {'bedroom' in context['questions_and_answers'].lower()}")
    print(f"Q&A contains 'quiet': {'quiet' in context['questions_and_answers'].lower()}")
    print(f"Q&A contains 'schools': {'schools' in context['questions_and_answers'].lower()}")
    
    listing_text = f"""
        PROPERTY DETAILS:
        • Neighborhood: {listing.neighborhood}
        • Price: €{listing.price:,}
        • Bedrooms: {listing.bedrooms}
        • Bathrooms: {listing.bathrooms}
        • Size: {listing.size_sqft:,} sqft
        
        Create a personalized description that mentions the buyer's specific preferences.
    """
    
    response = recommender_chain.run(
        recommendation_summary=context["recommendation_summary"],
        questions_and_answers=context["questions_and_answers"],
        input=listing_text,
    )
    
    print(f"\n📝 RESPONSE FOR {listing.neighborhood}:")
    
    # Checking for personalization keywords
    personalization_keywords = ["based on", "since you", "your preference", "you mentioned", "you want", "you need", "for you"]
    found_keywords = [kw for kw in personalization_keywords if kw in response.lower()]
    
    if found_keywords:
        print(f"✅ PERSONALIZED! Found keywords: {found_keywords[:3]}")
    else:
        print("⚠️ WARNING: May not be personalized")
    
    print("-" * 40)
    print(response[:300] + "..." if len(response) > 300 else response)

✅ Sending 577 chars of Q&A to LLM
✅ Sending summary: BUYER WANTS:
• 2-bedroom, 1-bathroom home with spacious kitchen
• Quiet neighborhood with good schoo...

🔍 SENDING TO LLM FOR Mitte:
Q&A contains 'bedroom': True
Q&A contains 'quiet': True
Q&A contains 'schools': True

📝 RESPONSE FOR Mitte:
✅ PERSONALIZED! Found keywords: ['based on', 'since you', 'your preference']
----------------------------------------
Based on your preference for a 2-bedroom, 1-bathroom home with a spacious kitchen, I am excited to present to you this charming property in the desirable neighborhood of Mitte. Priced at €550,000, this cozy home offers a comfortable living space of 950 sqft, perfect for your needs.

Since you mentio...
✅ Sending 577 chars of Q&A to LLM
✅ Sending summary: BUYER WANTS:
• 2-bedroom, 1-bathroom home with spacious kitchen
• Quiet neighborhood with good schoo...

🔍 SENDING TO LLM FOR Prenzlauer Berg:
Q&A contains 'bedroom': True
Q&A contains 'quiet': True
Q&A contains 'schools': True

📝 